# 02 — Data Preparation (CRISP-DM Phase 3)
**Goal:** turn the raw SBA 7(a) FOIA files into a model-ready dataset, one decision at a time.
**Run order:** Restart kernel → Run All. Each section ends with a check so you can see what the step did.

| Step | What happens |
|---|---|
| 1 | Load + clean both files, combine once |
| 2 | Build the target (Paid in Full vs Charged Off) |
| 3 | Remove columns that leak the answer or identify the borrower |
| 4 | Fix invalid values |
| 5 | Engineer features |
| 6 | Train / test split |
| 7 | Impute, scale, encode (fit on train only) |
| 8 | Save |

## 0. Setup

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, joblib
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Notebook lives in notebooks/ ; data in data/raw/ and data/processed/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

FILES = ["FOIA_7a_FY2020_Present_asof_260630.csv", "FOIA_7a_FY2010_FY2019_asof_260630.csv"]
SEED = 42
pd.set_option("display.max_columns", 50, "display.width", 140)

## 1. Load, clean, combine
**Decisions**
- Strip spaces/commas from column names and text values (same cleaning as Assignment 1, so category labels match your report).
- Empty strings → real missing values (otherwise `""` hides as a fake category).
- The two files use **different date formats** (2020+ is `m/d/YYYY`, 2010–2019 is `YYYY-MM-DD`), so dates are parsed per file.
- Concatenate **once**, and assert the fiscal years do not overlap — this catches the double-concat bug from Assignment 1.

In [2]:
def load(fname, date_fmt):
    df = pd.read_csv(RAW / fname, low_memory=False)
    df.columns = df.columns.str.replace(r"[\s,]+", "", regex=True)
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].str.replace(r"[\s,]+", "", regex=True)
        df[c] = df[c].mask(df[c] == "")
    for c in ["AsOfDate", "ApprovalDate", "FirstDisbursementDate"]:
        parsed = pd.to_datetime(df[c], format=date_fmt, errors="coerce")
        assert (parsed.isna() & df[c].notna()).sum() == 0, f"{fname}:{c} has dates not in {date_fmt}"
        df[c] = parsed
    return df

new = load(FILES[0], "%m/%d/%Y")     # FY2020+
old = load(FILES[1], "%Y-%m-%d")     # FY2010-2019
assert old["ApprovalFY"].max() < new["ApprovalFY"].min(), "files overlap -> check downloads"

df = pd.concat([old, new], ignore_index=True)
print(f"{len(old):,} + {len(new):,} = {len(df):,} rows (Assignment 1 baseline: 934,089)")
SNAPSHOT = df["AsOfDate"].max()
print("Snapshot date:", SNAPSHOT.date())

545,751 + 388,338 = 934,089 rows (Assignment 1 baseline: 934,089)
Snapshot date: 2026-06-30


## 2. Target variable
`LoanStatus` has five codes in the real file. Only **PIF** (paid in full) and **CHGOFF** (charged off) have a final outcome:

| Code | Meaning | Use |
|---|---|---|
| PIF | Paid in full | model (0) |
| CHGOFF | Charged off | model (1) |
| EXEMPT | Disbursed, still active | exclude — outcome unknown |
| COMMIT / CANCLD | Never funded | exclude — no repayment outcome |

**Label direction:** `default = 1`. This flips Assignment 1 (where success = 1) so the minority class is the "positive" class and precision/recall/PR-AUC describe defaults. Correlation signs invert accordingly.

The code stops if SBA introduces an unexpected status.

In [3]:
known = {"PIF", "CHGOFF", "EXEMPT", "COMMIT", "CANCLD"}
unexpected = set(df["LoanStatus"].dropna().unique()) - known
assert not unexpected and df["LoanStatus"].notna().all(), f"New/missing LoanStatus: {unexpected}"

print(df["LoanStatus"].value_counts().to_frame("loans").assign(share=lambda t: (t.loans/len(df)).round(3)))

# Keep a copy of ALL disbursed loans for the censoring check in section 2b
disbursed = df[df["LoanStatus"].isin(["PIF", "CHGOFF", "EXEMPT"])].copy()

df = df[df["LoanStatus"].isin(["PIF", "CHGOFF"])].copy()
df["default"] = (df["LoanStatus"] == "CHGOFF").astype(int)
print(f"\nModeling rows: {len(df):,} | defaults: {df['default'].sum():,} ({df['default'].mean():.1%}) | imbalance {(1-df['default'].mean())/df['default'].mean():.1f}:1")

             loans  share
LoanStatus               
PIF         462922  0.496
EXEMPT      292150  0.313
CANCLD      116848  0.125
CHGOFF       41046  0.044
COMMIT       21123  0.023

Modeling rows: 503,968 | defaults: 41,046 (8.1%) | imbalance 11.3:1


### 2b. Censoring check (read this — it affects how you interpret results)
Because only *resolved* loans are modeled, long-term loans are under-represented (they haven't had time to resolve). `TermInMonths` — your strongest signal — may partly reflect *which loans have finished*, not *which are riskier*. The table shows what share of disbursed loans have resolved, by term and vintage. If the share is far lower for long terms, plan a sensitivity run on older vintages (`approval_fy <= 2019`) in the modeling phase.

In [4]:
disbursed["term_bucket"] = pd.cut(disbursed["TermInMonths"], [-1, 0, 60, 84, 120, 1000],
                                  labels=["0", "1-60", "61-84", "85-120", "121+"])
disbursed["vintage"] = np.where(disbursed["ApprovalFY"] <= 2019, "FY2010-19", "FY2020+")
disbursed["resolved"] = disbursed["LoanStatus"].isin(["PIF", "CHGOFF"])
(disbursed.groupby(["vintage", "term_bucket"], observed=True)["resolved"]
          .agg(loans="size", share_resolved="mean").round(3))

loans  share_resolved
vintage   term_bucket                        
FY2010-19 0               228           0.974
          1-60          93850           0.981
          61-84        149926           0.990
          85-120       129118           0.844
          121+         105841           0.748
FY2020+   0                11           0.818
          1-60          27608           0.498
          61-84         22233           0.378
          85-120       198689           0.191
          121+          68614           0.219

## 3. Drop leakage and identifier columns
- **Leakage (these *are* the answer):** `LoanStatus`, `PaidInFullDate`, `ChargeOffDate`, `GrossChargeOffAmount`. Keeping any gives a near-perfect, meaningless model.
- **Identifiers / high-cardinality admin fields:** borrower & bank names, addresses, ZIPs, county, district, bank ID numbers, franchise name/NAICS description. No generalizable signal; also keeps borrower names out of the shared dataset.
- **Constant:** `AsOfDate`, `Program`.
- `LocationID` is **kept temporarily** — only to group the train/test split (it repeats, so it is *not* a unique loan ID).
- `BankName` and the bank ID columns are used in section 5 to build lender features first, then dropped.

## 4. Invalid values → missing

- `TermInMonths == 0` and `InitialInterestRate == 0` are not real values (0-month term, 0% commercial rate) → set to missing and imputed later, **on train only**.
- Guarantee ratio outside (0, 1] is impossible → missing.

In [5]:
for col in ["TermInMonths", "InitialInterestRate"]:
    n = (df[col] == 0).sum()
    df.loc[df[col] == 0, col] = np.nan
    print(f"{col}: {n:,} zeros set to missing")

TermInMonths: 231 zeros set to missing
InitialInterestRate: 26 zeros set to missing


## 5. Feature engineering
Each feature below states *what* and *why*.

**5.1 Loan size → `log_gross_approval`.** Heavily right-skewed; log keeps a few huge loans from dominating.

In [6]:
df["log_gross_approval"] = np.log1p(df["GrossApproval"])
print(f"skew raw {df['GrossApproval'].skew():.2f} -> log {df['log_gross_approval'].skew():.2f}")

skew raw 3.69 -> log 0.16


**5.2 SBA guarantee → `guarantee_pct`.** `GrossApproval` and `SBAGuaranteedApproval` correlate at r = 1.00. Keep loan size plus the *ratio* (the part that adds information), drop the dollar guarantee.

In [7]:
df["guarantee_pct"] = df["SBAGuaranteedApproval"] / df["GrossApproval"]
bad = (df["guarantee_pct"] <= 0) | (df["guarantee_pct"] > 1)
df.loc[bad, "guarantee_pct"] = np.nan
print(f"{bad.sum():,} impossible ratios set to missing; range {df['guarantee_pct'].min():.2f}-{df['guarantee_pct'].max():.2f}")

0 impossible ratios set to missing; range 0.07-1.00


**5.3 Jobs → `log_jobs_supported`.** Same reason as 5.1 (max in the thousands, median single digits).

In [8]:
df["log_jobs_supported"] = np.log1p(df["JobsSupported"])

**5.4 Approval-to-disbursement lag → `log_approval_lag_days`.** Known at origination time (not an outcome). Negative lags are date errors → missing.

In [9]:
lag = (df["FirstDisbursementDate"] - df["ApprovalDate"]).dt.days
print(f"{(lag < 0).sum():,} negative lags set to missing; {lag.isna().sum():,} blank dates")
df["log_approval_lag_days"] = np.log1p(lag.mask(lag < 0))

418 negative lags set to missing; 265 blank dates


**5.5 Industry → `naics_sector`.** ~1,150 six-digit NAICS codes → the 2-digit sector (codes 31–33, 44–45, 48–49 share a sector). Unknown codes become `Unknown`.

In [10]:
SECTORS = {"11":"Agriculture","21":"Mining","22":"Utilities","23":"Construction","31":"Manufacturing",
 "32":"Manufacturing","33":"Manufacturing","42":"Wholesale","44":"Retail","45":"Retail",
 "48":"Transportation","49":"Transportation","51":"Information","52":"Finance_Insurance",
 "53":"RealEstate","54":"Professional_Technical","55":"Management","56":"Admin_Support",
 "61":"Education","62":"Health_Care","71":"Arts_Entertainment","72":"Accommodation_Food",
 "81":"Other_Services","92":"Public_Admin"}
code6 = df["NaicsCode"].astype("Int64").astype("string").str.zfill(6)
df["naics_sector"] = code6.str[:2].map(SECTORS).fillna("Unknown").astype(object)
print(f"{df['NaicsCode'].nunique():,} codes -> {df['naics_sector'].nunique()} sectors")
df["naics_sector"].value_counts()

1,323 codes -> 21 sectors


naics_sector
Retail                    67748
Accommodation_Food        63587
Construction              56214
Professional_Technical    50017
Health_Care               45119
Other_Services            43488
Manufacturing             39134
Transportation            29482
Wholesale                 26854
Admin_Support             24395
Arts_Entertainment        14489
RealEstate                11254
Finance_Insurance          8999
Agriculture                8146
Education                  6801
Information                5867
Mining                     1479
Utilities                   489
Management                  305
Public_Admin                 99
Unknown                       2
Name: count, dtype: int64

**5.6 Lender → `lender_type`, `lender_volume_tier`.** Lender characteristics are part of the research question, but names are too high-cardinality.
- `lender_type`: bank (FDIC ID present) / credit union (NCUA ID present) / other.
- `lender_volume_tier`: size of the lender by *number of loans in the full file*. Uses no outcome information, so no leakage.

In [11]:
df["lender_type"] = np.select([df["BankFDICNumber"].notna(), df["BankNCUANumber"].notna()],
                              ["FDIC_bank", "NCUA_credit_union"], default="Other")
lender_n = pd.concat([old, new])["BankName"].value_counts()     # all loans, not just resolved
df["lender_volume_tier"] = pd.cut(df["BankName"].map(lender_n), [0, 99, 999, 4999, np.inf],
        labels=["lt_100", "100_999", "1000_4999", "5000_plus"]).astype(object)
print(df["lender_type"].value_counts().to_dict())
print(df["lender_volume_tier"].value_counts(dropna=False).to_dict())

{'FDIC_bank': 466606, 'Other': 20456, 'NCUA_credit_union': 16906}
{'5000_plus': 275662, '1000_4999': 103472, '100_999': 96814, 'lt_100': 28020}


**5.7 Franchise → `is_franchise` (Y/N).** Franchise code exists only for franchised borrowers; the flag keeps the signal.

In [12]:
df["is_franchise"] = np.where(df["FranchiseCode"].notna(), "Y", "N")

**5.8 `BusinessAge` kept as-is.** Assignment 1 found overlapping category definitions that cannot be untangled from the data, so merging would be guesswork and would hide the high default rate for new businesses. Missing becomes its own `Missing` level in section 7.

**5.9 Held out of the model but kept in the table** (for sensitivity checks, not as predictors):
- `approval_fy` — vintage; partly reflects how the sample was selected (see 2b).
- `matured` — 1 if the scheduled term has ended by the snapshot date.
- `sold_secondary_market` — recorded after origination; unclear meaning, so excluded from the baseline.

In [13]:
df["approval_fy"] = df["ApprovalFY"]
df["matured"] = ((df["ApprovalDate"] + pd.to_timedelta(df["TermInMonths"] * 30.4375, unit="D")) <= SNAPSHOT).astype(int)
df["sold_secondary_market"] = np.where(df["SoldSecMrktInd"].eq("Y"), "Y", "N")
for c in ["RevolverStatus", "CollateralInd"]:                    # only Y/N allowed
    df.loc[~df[c].isin(["Y", "N"]), c] = np.nan

### 5.10 Keep only what we need (drops all leakage / identifier / redundant columns)

In [14]:
NUM = ["log_gross_approval", "guarantee_pct", "TermInMonths", "InitialInterestRate",
       "log_jobs_supported", "log_approval_lag_days"]
CAT = ["naics_sector", "BusinessType", "BusinessAge", "ProjectState", "ProcessingMethod",
       "FixedorVariableInterestInd", "RevolverStatus", "CollateralInd", "is_franchise",
       "lender_type", "lender_volume_tier"]
HOLD = ["approval_fy", "matured", "sold_secondary_market"]
data = df[["LocationID", "default"] + NUM + CAT + HOLD].reset_index(drop=True)

banned = {"LoanStatus", "PaidInFullDate", "ChargeOffDate", "GrossChargeOffAmount"}
assert not banned & set(data.columns), "leakage column slipped in"
print(f"{data.shape[0]:,} rows x {data.shape[1]} columns")
data.isna().sum()[lambda s: s > 0].to_frame("missing").assign(pct=lambda t: (t.missing/len(data)*100).round(2))

503,968 rows x 22 columns


,missing,pct
TermInMonths,231,0.05
InitialInterestRate,26,0.01
log_approval_lag_days,683,0.14
BusinessType,19,0.00
BusinessAge,1343,0.27


## 6. Train / test split
- **80/20**, **stratified** on `default` so both sets keep the same default rate.
- **Grouped by `LocationID`**: it repeats across loans, and the same borrower location in both train and test would inflate scores.
- The test set is **sealed** — used once, at the end of modeling. Model selection uses cross-validation inside train.

In [15]:
group = data["LocationID"].copy()
group[group.isna()] = -(np.arange(group.isna().sum()) + 1)          # unique group per row with no ID
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx, te_idx = next(sgkf.split(data, data["default"], groups=group))

assert not set(group.iloc[tr_idx]) & set(group.iloc[te_idx]), "group leakage"
train, test = data.iloc[tr_idx].copy(), data.iloc[te_idx].copy()
print(f"train {len(train):,} ({train['default'].mean():.2%} default) | test {len(test):,} ({test['default'].mean():.2%} default)")

train 403,162 (8.14% default) | test 100,806 (8.15% default)


## 7. Impute, scale, encode — fit on TRAIN only
- **Numeric:** median imputation (+ a missing-indicator column), then standardize (needed for logistic regression, harmless for trees).
- **Categorical:** missing → `"Missing"`, one-hot encode; categories with < 500 train rows are pooled into one `infrequent` bin (e.g., tiny territories).
- Everything is learned from **train only** and then applied to test, so nothing from test leaks into training.

In [16]:
def frame(d):
    x = d[NUM + CAT].copy()
    for c in CAT: x[c] = x[c].astype(object).where(x[c].notna(), np.nan)
    for c in NUM: x[c] = x[c].astype(float)
    return x

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                      ("scale", StandardScaler())]), NUM),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Missing")),
                      ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                               min_frequency=500, sparse_output=True))]), CAT),
], sparse_threshold=1.0)

X_train = sparse.csr_matrix(pre.fit_transform(frame(train)))
X_test  = sparse.csr_matrix(pre.transform(frame(test)))
y_train, y_test = train["default"].to_numpy(), test["default"].to_numpy()
names = pre.get_feature_names_out()

assert np.isfinite(X_train.data).all() and np.isfinite(X_test.data).all()
assert X_train.shape[1] == X_test.shape[1] == len(names)
print("X_train", X_train.shape, "| X_test", X_test.shape)

X_train (403162, 126) | X_test (100806, 126)


AttributeError: 'csr_matrix' object has no attribute 'head'

## 8. Save

In [17]:
data.assign(split=np.where(data.index.isin(tr_idx), "train", "test")).to_csv(PROC / "analytic_table.csv", index=False)
sparse.save_npz(PROC / "X_train.npz", X_train); sparse.save_npz(PROC / "X_test.npz", X_test)
np.save(PROC / "y_train.npy", y_train);          np.save(PROC / "y_test.npy", y_test)
joblib.dump(pre, PROC / "preprocessor.joblib")
pd.Series(names, name="feature").to_csv(PROC / "feature_names.csv", index=False)
print("Saved to", PROC)

Saved to c:\Users\Mason\OneDrive\Desktop\smb_loan\data\processed


## Summary of decisions
| # | Decision | Reason |
|---|---|---|
| 1 | Parse two date formats separately; concat once with non-overlap check | Prevents silent NaT and double-counting |
| 2 | Model PIF vs CHGOFF only | Only these have a final outcome |
| 3 | `default = 1` | Minority class is positive for PR-AUC / recall |
| 4 | Drop outcome-revealing columns | Leakage |
| 5 | Drop names, addresses, IDs, constants | No signal / privacy |
| 6 | Zero term / zero rate → missing | Not real values |
| 7 | Log loan size and jobs | Right skew |
| 8 | `guarantee_pct` instead of guarantee $ | r = 1.00 collinearity |
| 9 | NAICS → 2-digit sector | 1,150 codes too sparse |
| 10 | Lender type + size tier | Lender effects without high-cardinality names |
| 11 | `BusinessAge` kept native | Overlapping definitions can't be merged reliably |
| 12 | Grouped, stratified 80/20 split | No location leakage, same default rate |
| 13 | Preprocess fit on train only | No test leakage |
| 14 | No resampling here | Handle imbalance inside CV on train |

**Carry into modeling:** (a) censoring risk from 2b → run a sensitivity model on older vintages; (b) expect modest accuracy — no borrower financials in this data; (c) evaluate with ROC-AUC and PR-AUC, not accuracy.